# IsoGrowingNCA Direct ISONCA Baseline

Self-contained Colab-style direct-NCA baseline modeled after the ISONCA notebooks in the repository root.

This notebook intentionally does **not** use LPPN/SIREN. It trains a low-resolution isotropic growing NCA where the first four state channels are the RGBA output directly.

The training loop follows the ISONCA recipe more closely than the generic growing-image path:

- laplacian-only isotropic perception
- structured seed option
- direct fixed or rotation-invariant target loss
- trajectory difference regularization
- overflow control on the NCA state
- damage training from the persistent pool

Default setup follows the ISONCA structured-seed target scale:

- target: Noto emoji lizard
- target size: 48 px
- padding: 12 px
- NCA grid/output: 72 x 72
- decoder: none

Use this as a sanity check that the isotropic NCA itself can learn a simple morphology and retain equivariance. Use `isonca_chameleon_demo.ipynb` for the high-resolution chameleon LPPN comparison.


In [ ]:
#@title Clone repository (Colab setup)
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/IvanLudvig/Cells2Pixels.git'
BRANCH = 'isotropic'
REPO_DIR = 'Cells2Pixels'

if not Path('train.py').exists():
    if not Path(REPO_DIR).exists():
        cmd = ['git', 'clone']
        if BRANCH:
            cmd += ['--branch', BRANCH]
        cmd += [REPO_URL, REPO_DIR]
        subprocess.run(cmd, check=True)
    os.chdir(REPO_DIR)

print('cwd:', Path.cwd())


In [ ]:
#@title Imports and device setup
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import clear_output
from PIL import Image

from losses.image_loss import imread
from models.isonca import IsoGrowingNCA
from training.common import (
    make_grad_scaler,
    normalize_model_grads,
    optimizer_scheduler_step,
    set_seed,
)
from utils.misc import autocast_context

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this notebook. Enable a GPU runtime in Colab.')

device = torch.device('cuda:0')
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
#@title Target image
TARGET_EMOJI = '🦎'  #@param {type: 'string'}
TARGET_IMAGE_PATH = None

emoji_code = hex(ord(TARGET_EMOJI[0]))[2:].lower()
TARGET_URL = f'https://github.com/googlefonts/noto-emoji/blob/main/png/128/emoji_u{emoji_code}.png?raw=true'
print('Target emoji URL:', TARGET_URL)


In [ ]:
#@title Experiment settings
SEED = 43
PRECISION = torch.float32  # fp32 is the stable default for direct ISONCA training.
OUTPUT_TYPE = 's'

EPOCHS = 15000  #@param {type: 'integer'}
BATCH_SIZE = 8  #@param {type: 'integer'}
POOL_SIZE = 256  #@param {type: 'integer'}
STEP_RANGE = (64, 96)
INJECT_SEED_INTERVAL = 32
SUMMARY_INTERVAL = 100
DAMAGE_INTERVAL = 6

CHANNELS = 16
FC_DIM = 128
UPDATE_PROB = 0.5
SCALE_FACTOR = 1
IMAGE_SIZE = (48, 48)
PADDING = (12, 12)

# Structured seeds match the root ISONCA structured-seed notebook. Set SEED_POINTS=1 for a single center seed.
SEED_POINTS = 2  #@param {type: 'integer'}
SEED_RADIUS = 4  #@param {type: 'integer'}

# `invariant` is the right choice for the laplacian-only scalar model on asymmetric targets.
LOSS_MODE = 'invariant'  #@param ['fixed', 'invariant']
MIRROR_INVARIANT = True
SHARPEN_TARGET = True
FIXED_LOSS_SCALE = 1000.0
DIFF_LOSS_WEIGHT = 10.0
OVERFLOW_LOSS_WEIGHT = 1.0
OVERFLOW_CLAMP = 2.0
LOWER_LR = 1e-5
UPPER_LR = 1e-3

EVAL_STEPS = 512
ANGLES = [30.0, 45.0, 60.0, 120.0]
RUN_VARIANTS = ['direct']

set_seed(SEED)


In [ ]:
#@title Build/train helpers
def make_target_tensor():
    target_np = imread(TARGET_URL, max_size=IMAGE_SIZE, mode='RGBA', center_crop=False)
    target_np[..., :3] *= target_np[..., 3:4]
    target = torch.tensor(target_np, device=device, dtype=torch.float32).permute(2, 0, 1)
    target = F.pad(target, [PADDING[1], PADDING[1], PADDING[0], PADDING[0], 0, 0])
    return target


def rgb_linspace(n):
    if n <= 1:
        return torch.ones(1, 3, device=device, dtype=torch.float32)
    hues = torch.linspace(0.0, 1.0, n + 1, device=device)[:-1]
    # Small HSV-to-RGB implementation for visually distinct structured seed colors.
    h = hues * 6.0
    i = torch.floor(h).to(torch.long) % 6
    f = h - torch.floor(h)
    q = 1.0 - f
    zeros = torch.zeros_like(h)
    ones = torch.ones_like(h)
    table = torch.stack([
        torch.stack([ones, f, zeros], -1),
        torch.stack([q, ones, zeros], -1),
        torch.stack([zeros, ones, f], -1),
        torch.stack([zeros, q, ones], -1),
        torch.stack([f, zeros, ones], -1),
        torch.stack([ones, zeros, q], -1),
    ])
    return table[i, torch.arange(n, device=device)]


def make_seed(model, n, h, w, seed_points=SEED_POINTS, seed_radius=SEED_RADIUS):
    x = torch.zeros(n, model.channels, h, w, device=device, dtype=PRECISION)
    if seed_points <= 1:
        x[:, 3:, h // 2, w // 2] = 1.0
        return x

    angles = torch.linspace(0.0, 2.0 * torch.pi, seed_points + 1, device=device)[:-1]
    ys = torch.round(h // 2 + seed_radius * torch.sin(angles)).to(torch.long).clamp(0, h - 1)
    xs = torch.round(w // 2 + seed_radius * torch.cos(angles)).to(torch.long).clamp(0, w - 1)
    colors = rgb_linspace(seed_points).to(dtype=PRECISION)
    x[:, :3, ys, xs] = colors.T[None]
    x[:, 3:, ys, xs] = 1.0
    return x


def make_circle_masks(n, h, w):
    xs = torch.linspace(-1.0, 1.0, w, device=device)[None, None, :]
    ys = torch.linspace(-1.0, 1.0, h, device=device)[None, :, None]
    center = torch.rand(2, n, 1, 1, device=device) - 0.5
    radius = 0.1 + 0.3 * torch.rand(n, 1, 1, device=device)
    xs = (xs - center[0]) / radius
    ys = (ys - center[1]) / radius
    return (xs * xs + ys * ys < 1.0).to(PRECISION)


def sharpen_filter(img):
    if not SHARPEN_TARGET:
        return img
    channels = img.shape[1]
    kernel_1d = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0], device=img.device, dtype=img.dtype)
    kernel_2d = (kernel_1d[:, None] * kernel_1d[None, :]) / 256.0
    kernel = kernel_2d.expand(channels, 1, 5, 5)
    blurred = F.conv2d(F.pad(img, [2, 2, 2, 2], mode='replicate'), kernel, groups=channels)
    return img + (img - blurred) * 2.0


class DirectISONCALoss:
    def __init__(self, target):
        self.target = target[None].to(device=device, dtype=torch.float32)
        self.grid_size = tuple(target.shape[-2:])
        h, w = self.grid_size
        assert h == w, 'The invariant loss expects a square target grid.'
        r = torch.linspace(0.5 / w, 1.0, w // 2, device=device)[:, None]
        angle = torch.arange(0.0, w * np.pi, 1.0, device=device, dtype=torch.float32) / (w / 2)
        self.polar_xy = torch.stack([r * angle.cos(), r * angle.sin()], -1)[None]
        polar_target = F.grid_sample(sharpen_filter(self.target), self.polar_xy, mode='bilinear', align_corners=False)
        self.fft_target = torch.fft.rfft(polar_target).conj()
        self.polar_target_sqnorm = polar_target.square().sum(-1, keepdim=True)

    def fixed_loss(self, x):
        return FIXED_LOSS_SCALE * (x[:, :4].to(torch.float32) - self.target[:, :4]).square().mean()

    def invariant_loss(self, x):
        batch = sharpen_filter(x[:, :4].to(torch.float32))
        polar_batch = F.grid_sample(
            batch,
            self.polar_xy.repeat(len(batch), 1, 1, 1),
            mode='bilinear',
            align_corners=False,
        )
        x_fft = torch.fft.rfft(polar_batch)
        corr = torch.fft.irfft(x_fft * self.fft_target, n=polar_batch.shape[-1])
        if MIRROR_INVARIANT:
            corr = torch.cat([corr, torch.fft.irfft(x_fft * self.fft_target.conj(), n=polar_batch.shape[-1])], -1)
        xx = polar_batch.square().sum(-1, keepdim=True)
        sqdiff = xx + self.polar_target_sqnorm - 2.0 * corr
        return sqdiff.mean([1, 2]).min(-1)[0].mean()

    def target_loss(self, x):
        if LOSS_MODE == 'fixed':
            return self.fixed_loss(x)
        if LOSS_MODE == 'invariant':
            return self.invariant_loss(x)
        raise ValueError(f'Unknown LOSS_MODE: {LOSS_MODE}')

    def make_summary(self, generated):
        target = self.target[0, :4].detach().cpu().permute(1, 2, 0).numpy()
        generated = generated[0, :4].detach().to(torch.float32).cpu().permute(1, 2, 0).numpy()
        diff = np.abs(np.clip(generated, 0.0, 1.0) - np.clip(target, 0.0, 1.0))
        return np.concatenate([target, np.clip(generated, 0.0, 1.0), diff], axis=1)


def build_variant(name):
    assert name == 'direct'
    model = IsoGrowingNCA(
        channels=CHANNELS,
        fc_dim=FC_DIM,
        update_prob=UPDATE_PROB,
        device=device,
        precision=PRECISION,
    ).to(device)
    target = make_target_tensor()
    loss_fn = DirectISONCALoss(target)
    return model, None, loss_fn, None, loss_fn.grid_size


@torch.no_grad()
def render_rgba(model, siren, renderer, state, perception=None):
    x_up = F.interpolate(state.to(torch.float32), scale_factor=SCALE_FACTOR, mode='bilinear')
    mask = model.get_living_mask(x_up).float()
    return (x_up[:, :4] * mask).permute(0, 2, 3, 1)


def composite_rgba(image):
    rgb = image[..., :3]
    alpha = image[..., 3:4]
    return rgb * alpha + (1.0 - alpha)


In [ ]:
#@title Train direct baseline
def train_variant(name):
    set_seed(SEED)
    model, siren, loss_fn, renderer, grid_size = build_variant(name)
    with torch.no_grad():
        pool = make_seed(model, POOL_SIZE, grid_size[0], grid_size[1])

    optimizer = torch.optim.Adam(model.parameters(), lr=UPPER_LR)
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        LOWER_LR,
        UPPER_LR,
        step_size_up=2000,
        mode='triangular2',
        cycle_momentum=False,
    )
    scaler = make_grad_scaler(device, PRECISION)
    loss_history = []
    skipped_steps = 0

    model.train()
    for epoch in range(EPOCHS + 1):
        with torch.no_grad():
            batch_idx = np.random.choice(len(pool), BATCH_SIZE, replace=False)
            x = pool[batch_idx]
            if epoch % INJECT_SEED_INTERVAL == 0:
                x[:1] = make_seed(model, 1, grid_size[0], grid_size[1])
            if DAMAGE_INTERVAL and epoch % DAMAGE_INTERVAL == 0:
                damage_mask = 1.0 - make_circle_masks(1, grid_size[0], grid_size[1])[:, None]
                x[-1:] *= damage_mask

        step_n = np.random.randint(*STEP_RANGE)
        z = None
        overflow_loss = x.new_tensor(0.0, dtype=torch.float32)
        diff_loss = x.new_tensor(0.0, dtype=torch.float32)
        with autocast_context(device, PRECISION):
            for _ in range(step_n):
                prev_x = x
                x, z = model(x)
                diff_loss = diff_loss + (x.to(torch.float32) - prev_x.to(torch.float32)).abs().mean()
                scalar_x = x[:, :CHANNELS].to(torch.float32)
                overflow_loss = overflow_loss + (scalar_x - scalar_x.clamp(-OVERFLOW_CLAMP, OVERFLOW_CLAMP)).square().mean()

        target_loss = loss_fn.target_loss(x)
        diff_loss = diff_loss / step_n
        overflow_loss = overflow_loss / step_n
        loss = target_loss + DIFF_LOSS_WEIGHT * diff_loss + OVERFLOW_LOSS_WEIGHT * overflow_loss

        if not torch.isfinite(loss) or not torch.isfinite(x).all():
            skipped_steps += 1
            with torch.no_grad():
                pool[batch_idx] = make_seed(model, BATCH_SIZE, grid_size[0], grid_size[1])
            optimizer.zero_grad(set_to_none=True)
            print(f'{name}: skipped non-finite step at epoch {epoch} (skipped={skipped_steps})')
            continue

        if PRECISION == torch.float16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        with torch.no_grad():
            normalize_model_grads(model)
            optimizer_scheduler_step(optimizer, scheduler, scaler, PRECISION)
            optimizer.zero_grad(set_to_none=True)
            pool[batch_idx] = x.detach()

        loss_history.append(loss.item())
        return_summary = epoch % SUMMARY_INTERVAL == 0
        if return_summary:
            clear_output(wait=True)
            summary_image = loss_fn.make_summary(x)
            fig, axes = plt.subplots(2, 1, figsize=(7, 10))
            axes[0].imshow(summary_image)
            axes[0].set_title(f'{name}: target/generated/difference - epoch {epoch}')
            axes[0].axis('off')
            axes[1].plot(loss_history, '.', alpha=0.25)
            axes[1].set_yscale('log')
            axes[1].set_title(f'loss = {loss_history[-1]:.4f}')
            plt.tight_layout()
            plt.show()
            print(
                f'{name}: epoch {epoch}/{EPOCHS} | loss: {loss_history[-1]:.6g} | '
                f'target: {target_loss.item():.6g} | diff: {diff_loss.item():.6g} | overflow: {overflow_loss.item():.6g}'
            )

    out_dir = Path('notebook_runs') / 'isonca_direct_baseline'
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out_dir / 'model.pth')
    print('saved:', out_dir)
    return {
        'name': name,
        'model': model.eval(),
        'siren': None,
        'loss_fn': loss_fn,
        'renderer': None,
        'grid_size': grid_size,
        'loss_history': loss_history,
        'skipped_steps': skipped_steps,
        'path': out_dir,
    }


variants = {'direct': train_variant('direct')}


In [ ]:
#@title Equivariance metrics
def rollout(model, x, steps):
    z = None
    with autocast_context(device, PRECISION):
        for _ in range(steps):
            x, z = model(x)
    return x, z


def rel_l2(a, b):
    return torch.linalg.vector_norm((a - b).reshape(a.shape[0], -1), dim=1) / (
        torch.linalg.vector_norm(b.reshape(b.shape[0], -1), dim=1) + 1e-8
    )


def affine_rotate_nchw(x, angle_deg):
    angle = torch.tensor(angle_deg * torch.pi / 180.0, device=x.device, dtype=x.dtype)
    c = torch.cos(angle)
    s = torch.sin(angle)
    theta = torch.zeros(x.shape[0], 2, 3, device=x.device, dtype=x.dtype)
    theta[:, 0, 0] = c
    theta[:, 0, 1] = -s
    theta[:, 1, 0] = s
    theta[:, 1, 1] = c
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=False)


def affine_rotate_nhwc(image, angle_deg):
    x = image.permute(0, 3, 1, 2)
    x = affine_rotate_nchw(x, angle_deg)
    return x.permute(0, 2, 3, 1)


@torch.no_grad()
def evaluate_variant(variant, steps=EVAL_STEPS, angles=ANGLES):
    model = variant['model']
    siren = variant['siren']
    renderer = variant['renderer']
    old_update_prob = model.update_prob
    model.update_prob = 1.0
    rows = []
    visuals = {}
    try:
        h, w = variant['grid_size']
        x0 = make_seed(model, 1, h, w)
        x_base, z_base = rollout(model, x0, steps)
        image_base = render_rgba(model, siren, renderer, x_base, z_base)

        def add_result(kind, label, state, expected_state, image, expected_image):
            err = (image - expected_image).abs()
            rows.append({
                'kind': kind,
                'transform': label,
                'state_mae': (state - expected_state).abs().mean().item(),
                'state_rel_l2': rel_l2(state, expected_state).mean().item(),
                'render_mae': err.mean().item(),
                'render_rel_l2': rel_l2(image, expected_image).mean().item(),
            })
            visuals[label] = {
                'base': image_base.detach().cpu(),
                'transformed_seed': image.detach().cpu(),
                'transformed_output': expected_image.detach().cpu(),
                'error': err.detach().cpu(),
            }

        for k in (1, 2, 3):
            x_t0 = torch.rot90(x0, k, dims=(-2, -1))
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(model, siren, renderer, x_t, z_t)
            add_result('exact_rotation', f'rot{k * 90}', x_t, torch.rot90(x_base, k, dims=(-2, -1)), image_t, torch.rot90(image_base, k, dims=(-3, -2)))

        for label, dims in (('flip_x', (-1,)), ('flip_y', (-2,))):
            x_t0 = torch.flip(x0, dims=dims)
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(model, siren, renderer, x_t, z_t)
            image_dims = tuple(d - 1 for d in dims)
            add_result('reflection', label, x_t, torch.flip(x_base, dims=dims), image_t, torch.flip(image_base, dims=image_dims))

        for angle in angles:
            x_t0 = affine_rotate_nchw(x0, angle)
            x_t, z_t = rollout(model, x_t0, steps)
            image_t = render_rgba(model, siren, renderer, x_t, z_t)
            add_result('interpolated_rotation', f'rot{angle:g}', x_t, affine_rotate_nchw(x_base, angle), image_t, affine_rotate_nhwc(image_base, angle))
    finally:
        model.update_prob = old_update_prob
    return rows, visuals


results = {}
visuals = {}
for name, variant in variants.items():
    rows, vis = evaluate_variant(variant)
    results[name] = rows
    visuals[name] = vis
    print()
    print(name)
    print('kind | transform | state_mae | state_rel_l2 | render_mae | render_rel_l2')
    for row in rows:
        print(f"{row['kind']} | {row['transform']} | {row['state_mae']:.3e} | {row['state_rel_l2']:.3e} | {row['render_mae']:.3e} | {row['render_rel_l2']:.3e}")


In [ ]:
#@title Visual comparison grids
def show_grid(variant_name, transform='rot45'):
    item = visuals[variant_name][transform]
    titles = ['original render', 'rollout(transform(seed))', 'transform(rollout(seed))', 'absolute error']
    tensors = [item['base'], item['transformed_seed'], item['transformed_output'], item['error']]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, title, tensor in zip(axes, titles, tensors):
        img = tensor[0].numpy()
        if title == 'absolute error':
            ax.imshow(img.mean(axis=-1), cmap='magma')
        else:
            ax.imshow(np.clip(composite_rgba(img), 0.0, 1.0))
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{variant_name}: {transform}')
    plt.tight_layout()
    plt.show()


for name in visuals:
    show_grid(name, 'rot45')


In [ ]:
# Try other transforms after the previous cell runs:
# show_grid('direct', 'flip_x')
# show_grid('direct', 'rot90')
# show_grid('direct', 'rot60')
